In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl")
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi


#Lamb_shift = -0.01739466142645618 #Lamb shift in GHz
ω_b_0 = 2*ω_a #+ Lamb_shift*2*pi #7.7*2*pi

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive


ω_d=ω_b_0#-Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.8#
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=20.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = κ_b/100#3e-3*κ_2 #from Leon's paper

In [ ]:
#dict_freq_eff = Dict(0.2=>7.7846185068410705, 0.3=>7.87, 0.4=>7.95, 0.5=>8.035, 0.6 =>8.11, 0.7 => 8.175, 0.8 => 8.2315, 0.9 => 8.2754, 1 => 8.312, 1.1 => 8.3406, 1.2 => 8.364, 1.3 => 8.381, 1.4 => 8.391, 1.5 => 8.3972, 1.6 => 8.398) #epsilon_d = 10.12*g2, E_J=37*2*pi
dict_freq_eff = Dict(0.2=>7.867, 0.3=>8.005, 0.4=>8.1295, 0.5=>8.205772521909807, 0.6 =>8.3165, 0.7 => 8.382, 0.8 => 8.4334, 0.9 => 8.474, 1 => 8.506, 1.1 => 8.531, 1.2 => 8.5493, 1.3 => 8.563, 1.4 => 8.5725, 1.5 => 8.577, 1.6 => 8.578) #epsilon_d = 10.12*g2, E_J=37*2*pi*1.5
#dict_freq_eff = Dict(0.2=>8.055, 0.3=>8.247, 0.4=>8.385, 0.5=>8.483, 0.6 =>8.554, 0.7 => 8.606, 0.8 => 8.645, 0.9 => 0.0, 1 => 0.0, 1.1 => 0.0, 1.2 => 0.0, 1.3 => 0.0, 1.4 => 0.0, 1.5 => 0.0, 1.6 => 0.0) #epsilon_d = 10.12*g2, E_J=37*2*pi*2.5

In [ ]:
#ATS + Mean_Field + ladder_operator
function test_equation!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    sin_pump =  - 2*E_J*sin(ϵ_p*cos(ω_p*t))
    coupling = (cos((a+conj(a))/sqrt(2) * sqrt(Z_a)/ϕ_quantum + (b+conj(b))/sqrt(2) * sqrt(Z_b)/ϕ_quantum)) / ϕ_quantum / sqrt(2)
    
        
    du[1] = -1im*(ω_a*a + sqrt(Z_a)*sin_pump*coupling) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) + sqrt(Z_b)*sin_pump*coupling) - p.environment_prepared.κ_b * b
    return nothing
end

function find_peaks(x::AbstractVector; threshold = 0.0)
    n = length(x)
    peaks = Vector{Tuple{Float64,Int}}()

    @inbounds for i in 2:n-1
        if x[i] > x[i-1]+threshold && x[i] > x[i+1]+threshold
            push!(peaks, (x[i],i))
        end
    end

    return peaks
end

In [ ]:
t_max=100/κ_b
a_0 = 0.375
b_0 = 0.0

ω_d_eff = 8.645*2*pi#8.205772521909807*2*pi#ω_b_0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d_eff, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
a, b, t_list = simulator(t_max, system, drive, environment, equation = test_equation!);
F, freqs = FFT_perso(real.(b), t_list)


In [ ]:

plot(freqs, abs.(F), linewidth=.1)
#vline!([ω_d_eff/2/pi], label=L"ω_{d,eff}/2pi", ls=:dash)
vline!([8.645], ls=:dash)
plot!(xlims=(7.8,9), yaxis=:log)

In [ ]:
mask = (freqs .> 7.6) .& (freqs .< 8.9)
F_m = F[mask]
freqs_m = freqs[mask]
peaks = find_peaks(abs.(F_m), threshold = 0.1)
print(freqs_m[getindex.(peaks,2)])

index_max = argmax(getindex.(peaks,1))
peaks[index_max] = (0.0, 0.0)
index_max = argmax(getindex.(peaks,1))
freqs_max = freqs_m[getindex.(peaks,2)[index_max]]


In [ ]:
F, freqs = FFT_perso(real.(a), t_list)
plot(freqs, abs.(F), linewidth=.1)
vline!([ω_a/2/pi], label=L"ω_a/2pi")
vline!([ω_d_eff/2/pi/2], label=L"ω_{d,eff}/2pi")
plot!(xlims=(3,4.5), yaxis=:log)

In [ ]:
mask = (freqs .> 3.5) .& (freqs .< 4.5)

freqs_m = freqs[mask]
F_m     = F[mask]

freq_max = abs(freqs_m[argmax(abs.(F_m))])
a_rot = (a.-sum(a)/length(a)).*exp.(1im*freq_max*2*pi*t_list);

In [ ]:
a_rot[end].+sum(a)/length(a)

In [ ]:
plot(real.(a_rot[1:div(length(a_rot),3)].+real(sum(a)/length(a))), imag.(a_rot[1:div(length(a_rot),3)]).+imag(sum(a)/length(a)), label="")
plot!(xlabel=L"\Re(a)", ylabel=L"\Im(a)", size = size_px_car, 
title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",)


In [ ]:
plot(real.(a_rot[1:end].+real(sum(a)/length(a))), imag.(a_rot[1:end]).+imag(sum(a)/length(a)))

In [ ]:
system_prepared = prepare(system)
drive_prepared = prepare(drive)
environment_prepared = prepare(system, environment)
u_0 = get_init(system_prepared, environment_prepared)
p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
print(p)
dt = 1/(34*sqrt(2)*ω_a)
a, b, t_list = default_solver(u_0, (0.0, 100/κ_b), p, dt, equation = test_equation!)
a_rot_bis = (a.-sum(a)/length(a)).*exp.(1im*freq_max*2*pi*t_list);

In [ ]:
length(a_rot_bis)

In [ ]:
plot(real.(a_rot_bis[1:end].+real(sum(a)/length(a))), imag.(a_rot_bis[1:end]).+imag(sum(a)/length(a)))

# Poincaré sections

First we need to get the stroboscopic frequencies : freqs_max_a

In [ ]:
ϵ_p = 0.2
t_max = 200/κ_b
t_min = 0.0
t_min_save = 0.0
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=1/100
t_save = max(t_min_save,0):dt_save:t_max
a_0 = 0.375
b_0 = 0.0

freqs_max_a_vec = Vector{Float64}()
freqs_max_b_vec = Vector{Float64}()

freq_window_start = ω_b_0/2pi + 0.05
freq_window_stop = freq_window_start + 0.2

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
environment = MeanField(κ_a, κ_b)
system_prepared = prepare(system)
environment_prepared = prepare(system,environment);

ω_d_eff = ω_b_0
g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=20.12*g2
drive = DriveTwoModesBasics(ϵ_d, ω_d_eff, buffer_phase, ϵ_p, ω_p, pump_phase)
drive_prepared = prepare(drive)

u_0 = get_init(system_prepared, environment_prepared)
p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

_, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = test_equation!)

F, freqs = FFT_perso(real.(b), t_list)
mask = (freqs .> freq_window_start) .& (freqs .< freq_window_stop)
freqs_m = freqs
F_m     = F;

In [ ]:
plot(freqs_m, abs.(F_m), linewidth=.1)
plot!(yaxis=:log, xlims=(5,10))

In [ ]:
length(ϵ_p_list)

In [ ]:
ϵ_p_list = 0.2:0.05:0.85
t_max = 400/κ_b
t_min = 0.0
t_min_save = 0.0
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=1/100
t_save = max(t_min_save,0):dt_save:t_max
a_0 = 0.375
b_0 = 0.0

freqs_max_a_vec = Vector{Float64}()
freqs_max_b_vec = Vector{Float64}()

freq_window_start = ω_b_0/2pi + 0.05
freq_window_stop = freq_window_start + 0.2

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
environment = MeanField(κ_a, κ_b)
system_prepared = prepare(system)
environment_prepared = prepare(system,environment);
for ϵ_p in ϵ_p_list
    ω_d_eff = ω_b_0
    g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=20.12*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d_eff, buffer_phase, ϵ_p, ω_p, pump_phase)
    drive_prepared = prepare(drive)

    u_0 = get_init(system_prepared, environment_prepared)
    p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

    _, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = test_equation!)

    F, freqs = FFT_perso(real.(b), t_list)
    mask = (freqs .> freq_window_start) .& (freqs .< freq_window_stop)
    freqs_m = freqs[mask]
    F_m     = F[mask]
    peaks = find_peaks(abs.(F_m), threshold = 0.01)
    index_peak = argmax(getindex.(peaks,1))
    freq_max_b = freqs_m[getindex.(peaks,2)[index_peak]]

    append!(freqs_max_b_vec, freq_max_b)
    freq_window_start = freq_max_b
    freq_window_stop = freq_max_b+0.1
    
    
    ω_d_eff = 2*pi*freq_max_b
    drive = DriveTwoModesBasics(ϵ_d, ω_d_eff, buffer_phase, ϵ_p, ω_p, pump_phase)
    a, _, t_list = simulator(t_max, system, drive, environment, equation = test_equation!)
    F, freqs = FFT_perso(real.(a), t_list)
    mask = (freqs .> 3.5) .& (freqs .< 4.5)
    freqs_m = freqs[mask]
    F_m     = F[mask]
    freq_max_a = freqs_m[argmax(abs.(F_m))]
    append!(freqs_max_a_vec, freq_max_a)
end

In [ ]:
freqs_max_b_vec

In [ ]:
t_max = 130/κ_b
t_min = 0.0

t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
environment = MeanField(κ_a, κ_b)
environment_prepared = environment

Im_a_list=range(-4,4,10)
Re_a_list=range(-10,10,25)
Final_all_a = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_b = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
#a_rot = Vector{ComplexF64}()
#mean_a = 0.0

Threads.@threads for idx in eachindex(ϵ_p_list)
    ϵ_p = ϵ_p_list[idx]
    g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d = 20.12*g2
    ω_d_eff = freqs_max_b_vec[idx]*2*pi
    ω_a_eff = freqs_max_b_vec[idx]*2*pi

    dt = 1 / (50 * ω_a) # Time step for the simulation
    dt_save = 2pi/(ω_a_eff)
    t_save = max(t_min_save,0):dt_save:t_max
    drive = DriveTwoModesBasics(ϵ_d, ω_d_eff, buffer_phase, ϵ_p, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j
            system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = test_equation!)

            Final_mode_a[(re_i,im_j)] = a
            Final_mode_b[(re_i,im_j)] = b
            #a_rot = (a.-sum(a)/length(a)).*exp.(1im*ω_a_eff*t_list);
            #mean_a = sum(a)/length(a)
        end
    end
    Final_all_a[ϵ_p] = Final_mode_a
    Final_all_b[ϵ_p] = Final_mode_b
end

In [ ]:
for ϵ_p in ϵ_p_list

    g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b
    Final_mode = Final_all_b[ϵ_p]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        z = Final_mode[(re_i, im_j)]
        scatter!(
            p0,
            real.(z),
            imag.(z),
            ms = 1,
            color = :black,
            label = "",
        )
    end

    # Add labels and title. 
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          top_margin=5mm)

    display(p0)
end


# We try to increase step by step the amplitudes of the drives

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi


#Lamb_shift = -0.01739466142645618 #Lamb shift in GHz
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi


##Drive


ω_d=ω_b#-Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
ϵ_0 = 0.2#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=8*g2*0

In [ ]:
t_max=50/κ_b
a_0 = 0.0
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesJA(ϵ_d, 7.8*2pi, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
a, b, t_list = simulator(t_max, system, drive, environment, equation = test_equation!);

In [ ]:
F, freqs = FFT_perso(real.(a), t_list)
plot(freqs, abs.(F), linewidth=.1)
vline!([ω_a/2/pi], ls=:dash, lw=.5)
plot!(xlims=(3.5,4.5), yaxis=:log)

In [ ]:
mask = (freqs .> 3.5) .& (freqs .< 4.5)

freqs_m = freqs[mask]
F_m     = F[mask]

freq_max_a = abs(freqs_m[argmax(abs.(F_m))])

In [ ]:
F, freqs = FFT_perso(real.(b), t_list)
plot(freqs, abs.(F), linewidth=.1)
vline!([ω_b/2/pi], ls=:dash, lw=.5)
vline!([2*freq_max_a], ls=:dash, lw=.5)
plot!(xlims=(7.5,8), yaxis=:log)

In [ ]:
2*freq_max_a

In [ ]:
mask = (freqs .> 7.0) .& (freqs .< 9.0)

freqs_m = freqs[mask]
F_m     = F[mask]

freq_max_b = abs(freqs_m[argmax(abs.(F_m))])

In [ ]:
ω_p_2 = (2*freq_max_a - freq_max_b)*2*pi

In [ ]:
ϵ_0 = 0.2#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=8*g2*0

t_max=50/κ_b
a_0 = 0.0
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesJA(ϵ_d, freq_max_b*2pi, buffer_phase, ϵ_0, ω_p_2, pump_phase)
environment = MeanField(κ_a, κ_b)
a, b, t_list = simulator(t_max, system, drive, environment, equation = test_equation!);

In [ ]:
F, freqs = FFT_perso(real.(b), t_list)
plot(freqs, abs.(F), linewidth=.1)
vline!([freq_max_b], ls=:dash, lw=.5)
plot!(xlims=(7,9), yaxis=:log)

In [ ]:
ϵ_p_list = 0.1:0.093:0.5pi
t_max = 50/κ_b
t_min = 0.0
tspan = (t_min, t_max) # Time span for the simulation

a_0 = 0.0
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, 0.0, E_Cb, a_0, b_0)
system_prepared = prepare(system)
environment = MeanField(κ_a, κ_b)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
dt_save = 40*dt
t_save = t_min:dt_save:t_max
fs = length(t_save)/(maximum(t_save)-minimum(t_save)) # sampling frequency
freqs = fftshift(fftfreq(length(t_save), fs))

FFT_dict_b = Dict{Float64, Vector{Float64}}()
freqs_max_b_vec = Vector{Float64}()
mask_b = (freqs .> 7.5) .& (freqs .< 9.5)
freqs_m_b = freqs[mask_b]

FFT_dict_a = Dict{Float64, Vector{Float64}}()
freqs_max_a_vec = Vector{Float64}()
mask_a = (freqs .> 3) .& (freqs .< 5)
freqs_m_a = freqs[mask_a]

for ϵ_p in ϵ_p_list
    g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
    drive = DriveTwoModesJA(0.0, 0.0, buffer_phase, ϵ_p, 0, pump_phase)
    drive_prepared = prepare(drive)
    u_0_tot = get_init(system_prepared, environment_prepared)
    p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
    a, b, _ = default_solver(u_0_tot, tspan, p, dt; equation=test_equation!, t_save = t_save)
    
    F_b = fftshift(fft(real.(b)))
    F_m_b = F_b[mask_b]
    freq_max_b = abs(freqs_m_b[argmax(abs.(F_m_b))])
    
    F_a = fftshift(fft(real.(a)))

    FFT_dict_b[ϵ_p] = abs.(F_b)
    append!(freqs_max_b_vec, freq_max_b)
    FFT_dict_a[ϵ_p] = abs.(F_a)
end

In [ ]:
plot(besselj.(1,ϵ_p_list)*E_J*φ_a^2*φ_b/2pi*1e3, freqs_max_b_vec)

In [ ]:
p0=plot()
for ϵ_p in ϵ_p_list
    plot!(p0, freqs, FFT_dict_b[ϵ_p], label="")
end
plot!(p0, xlims=(7.5,11.0), yaxis=:log)

In [ ]:
freqs_max

In [ ]:
ϵ_p = ϵ_p_list[12]
freqs_max = freqs_max_b_vec[12]

t_max = 50/κ_b
t_min = 0.0
tspan = (t_min, t_max) # Time span for the simulation

a_0 = 0.0
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
system_prepared = prepare(system)
environment = MeanField(κ_a, κ_b)
environment_prepared = prepare(system,environment)

g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d = 10.12*g2
drive = DriveTwoModesJA(ϵ_d, freqs_max*2*pi, buffer_phase, ϵ_p, 0.0, pump_phase)
drive_prepared = prepare(drive)
u_0_tot = get_init(system_prepared, environment_prepared)
p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
a, b, t_list = default_solver(u_0_tot, tspan, p, dt; equation=test_equation!, t_save = t_save)

F_b = fftshift(fft(real.(b)))
F_m_b = F_b[mask_b]
freq_max_b = abs(freqs_m_b[argmax(abs.(F_m_b))])

F_a = fftshift(fft(real.(a)))
F_m_a = F_b[mask_a]
freq_max_a = abs(freqs_m_a[argmax(abs.(F_m_a))])


In [ ]:
freqs_max

In [ ]:
plot(freqs, abs.(F_b))
plot!(xlims=(7,10), yaxis=:log)

In [ ]:
plot(freqs, abs.(F_a))
plot!(xlims=(3,5), yaxis=:log)

In [ ]:
a_rot =@. a*exp(1im*freq_max_a*2pi*t_list);

In [ ]:
plot(real.(a_rot[end-5000:end]), imag.(a_rot[end-5000:end]), lw=.2)

# Bistability

In [ ]:
t_max=50/κ_b
b_0 = 0.0
theta_range = (0:1:10)*2pi/10
a_plot_list = []
for theta in theta_range
    a_0 = 0.375 + 0.02*exp(1im*theta)
    system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
    drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    environment = MeanField(κ_a, κ_b)
    a, _, t_list = simulator(t_max, system, drive, environment, equation = test_equation!);
    a_rot = (a.-sum(a)/length(a)).*exp.(1im*freq_max*2*pi*t_list);
    a_plot = a_rot[1:div(length(a_rot),2)].+sum(a)/length(a)
    push!(a_plot_list,a_plot)
end

In [ ]:
p0 = plot()
for (i,theta) in enumerate(theta_range)
    a_plot = a_plot_list[i]
    plot!(real.(a_plot), imag.(a_plot))
end
display(p0)

In [ ]:
571*0.6+39

In [ ]:
585*0.6+39